# Temporal Patterns & Pandemic Analysis| | ||---|---|| **Project** | Vancouver Urban Safety Intelligence Platform (CSIS 4495) || **Owner** | Jaismine Kaur (Data Analytics workstream) || **Date** | October 11, 2026 || **Phase** | Initial implementation (Oct 7-12) |This notebook answers three questions:1. Does each crime type have its own time-of-day pattern?2. How did the COVID-19 pandemic change Vancouver's crime mix?3. Do seasons and neighbourhoods matter - and what happens when a hypothesis fails?The findings become the first content of `app/pages/4_statistical.py` in theOct 13-20 phase.## Contents1. Time-of-Day Patterns: Does Each Crime Type Have Its Own "Clock"?2. The COVID-19 Natural Experiment3. Seasonality Hypothesis4. Neighbourhood Change, 2019-20235. Summary & Next Steps

## Setup

In [ ]:
%matplotlib inlinefrom pathlib import Pathimport matplotlib.pyplot as pltimport pandas as pddf = pd.read_csv("../data/processed/crime_data_cleaned.csv")print(df.shape)df.head()

## 1. Time-of-Day Patterns: Does Each Crime Type Have Its Own "Clock"?We normalize by row so each crime type's hours add to 100%. This makes a raretype (Homicide) comparable with a common one (Theft from Vehicle).

In [ ]:
ct = pd.crosstab(df["TYPE"], df["HOUR"], normalize="index")print("Peak hour per crime type:")for t in ct.index:    peak = ct.loc[t].idxmax()    print(f"{t[:40]:40s} peak at {peak:02d}:00  ({ct.loc[t].max()*100:.1f}%)")

### 1.1 Data Forensics: Redacted TimestampsBefore drawing conclusions, check the raw hour values for the violentcategories.

In [ ]:
print(df[df["TYPE"].isin(["Homicide", "Offence Against a Person"])]["HOUR"].value_counts())

> **Finding 1 - Data forensics.** Every violent incident is stamped at> hour 00:00. This is deliberate: VPD redacts the *time* of violent crimes as> a privacy measure (the same reason violent types use broad categories).> Hour-of-day analysis must therefore exclude `Homicide` and `Offence Against> a Person`. The redaction itself is worth documenting in the final report.

### 1.2 Night vs Daytime Shares

In [ ]:
night = ct[[22, 23, 0, 1, 2, 3, 4, 5]].sum(axis=1) * 100day = ct[[9, 10, 11, 12, 13, 14, 15, 16, 17]].sum(axis=1) * 100print("Night (22-05) vs daytime (09-17) share per type:")for t in ct.index:    print(f"{t[:40]:40s} night {night[t]:4.1f}%   daytime {day[t]:4.1f}%")

### 1.3 Hour Profiles (one chart per crime type)

In [ ]:
for t in ct.index:    ct.loc[t].plot(figsize=(8, 2), title=t)    plt.xlabel("Hour"); plt.ylabel("Share of that type")    plt.tight_layout()    plt.show()

> **Finding 2 - Every crime has its own clock.** Commercial Break & Enter> is ~49% nocturnal (peak 04:00), while Other Theft is ~58% daytime (peak> 15:00). Collisions peak at 17:00 (rush hour). Crime type strongly predicts> *when* incidents happen - a natural candidate for the chi-square tests in> the next phase.

## 2. The COVID-19 Natural Experiment2022 is missing from the dataset, so we compare 2019 (pre-pandemic), 2020(lockdown), 2021, and 2023 (post-pandemic).

In [ ]:
years = df[df["YEAR"].isin([2019, 2020, 2021, 2023])]y = pd.crosstab(years["TYPE"], years["YEAR"])print(y)

In [ ]:
# Pandemic shock: 2019 -> 2020change = (y[2020] - y[2019]) / y[2019] * 100print("Biggest changes 2019 -> 2020 (%):")print(change.sort_values().round(1))

In [ ]:
# Post-pandemic: 2021 -> 2023change2 = (y[2023] - y[2021]) / y[2021] * 100print("Biggest changes 2021 -> 2023 (%):")print(change2.sort_values().round(1))

### 2.1 Visualization of the Big Movers

In [ ]:
y.T[["Theft from Vehicle", "Other Theft", "Mischief",      "Break and Enter Commercial"]].plot(marker="o", figsize=(10, 5))plt.title("Pandemic and post-pandemic shift")plt.ylabel("Incidents"); plt.grid(alpha=0.3)plt.tight_layout()plt.show()

> **Finding 3 - The pandemic reshaped the mix.** In 2020 the streets> emptied: collisions with injury -43%, Theft from Vehicle -39%, and Vehicle> Theft -38%, while Commercial Break & Enter rose +12% (shuttered businesses> became targets). By 2023 the mix changed again: Other Theft +50% and> Mischief +19% versus 2021, while bicycle theft and residential break-ins> kept falling. Recovery did not restore the old pattern.

## 3. Seasonality Hypothesis### Hypothesis A: "Theft from Vehicle is worse in summer"Reasoning: warmer weather, more people out, more cars parked with windowsdown. Let's test it against the all-crime seasonal baseline.

In [ ]:
tfv = df[df["TYPE"] == "Theft from Vehicle"]season_order = ["Winter", "Spring", "Summer", "Fall"]s = tfv["SEASON"].value_counts().reindex(season_order)print("Theft from Vehicle by season:")for season in season_order:    print(f"  {season:8s} {s[season]:,}  ({s[season]/len(tfv)*100:.1f}%)")base = df["SEASON"].value_counts().reindex(season_order) / len(df) * 100print("\nAll crime baseline (%):")print(base.round(1))

> **Finding 4 - The hypothesis that failed.** Theft from Vehicle is almost> perfectly flat across seasons (24.2 / 24.7 / 25.3 / 25.7%), even though all> crime has a small summer bump (26.5% in summer vs 23.6% in winter). The> hypothesis is *rejected* - and a rejected hypothesis is still a result worth> reporting. Seasonality is category-specific; the chi-square tests next week> can quantify exactly how much.

## 4. Neighbourhood Change, 2019-2023### Hypothesis B: "Some neighbourhoods improved more than others"Compare incident counts per neighbourhood between 2019 and 2023.

In [ ]:
hood = (df[df["YEAR"].isin([2019, 2023])]         .groupby(["NEIGHBOURHOOD", "YEAR"])         .size()         .unstack())hood["change_pct"] = (hood[2023] - hood[2019]) / hood[2019] * 100print("Top 5 improvers (% change 2019 -> 2023):")print(hood.nsmallest(5, "change_pct")[["change_pct"]].round(1))print("\nNeighbourhoods with increases:")print(hood[hood["change_pct"] > 0][["change_pct"]].round(1))

> **Finding 5 - Neighbourhoods diverge.** 21 of 24 neighbourhoods recorded> *fewer* incidents in 2023 than in 2019. The biggest improvers were Oakridge> (-50.7%), Riley Park (-46.2%), and Hastings-Sunrise (-42.7%). Only Musqueam> (+25.8%), Renfrew-Collingwood (+21.3%), and Sunset (+10.8%) increased.>> **Caveats:** Musqueam has a small population, so percentage swings are noisy;> 2022 is missing; and population shifts can explain part of these changes.> Normalizing by population/area is on the roadmap for the statistical phase.

## 5. Summary & Next Steps| # | Finding | Use in Oct 13-20 phase ||---|---|---|| 1 | Violent-crime timestamps are redacted (hour = 0) | Report + exclude from time analysis || 2 | Crime type predicts time-of-day patterns | Chi-square test on type x hour/season || 3 | COVID changed the crime mix; recovery uneven | Trend/statistical analysis || 4 | Theft from Vehicle has no seasonality | Documented rejected hypothesis || 5 | 21/24 neighbourhoods improved; 3 worsened | Neighbourhood profiles + map phase |**Next steps:**- Move Findings 2-5 into `app/pages/4_statistical.py` with proper statistical  tests (`scipy`).- Normalize neighbourhood comparisons by population/area in the Crime Map  phase.- Keep the redacted-timestamp caveat visible wherever hour-of-day results are  shown.